In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Tuple, Union

import lightgbm as lgbm
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb


def calculate_avm_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute headline (MdAPE) and complementary performance metrics on raw dollar scale.

    Args:
        y_true: Ground truth property close prices on raw dollar scale.
        y_pred: Model predicted close prices on raw dollar scale.

    Returns:
        Dict mapping metric names to their rounded floating-point values.
    """
    # Calculate absolute and relative percentage errors per property
    abs_errors = np.abs(y_true - y_pred)
    percentage_errors = abs_errors / y_true

    # Headline Metric: Robust to extreme skew and price scale divergence
    mdape = float(np.median(percentage_errors) * 100)

    # Complementary Metrics: Average percentage, dollar-denominated, and variance metrics
    mape = float(np.mean(percentage_errors) * 100)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))

    return {
        "MdAPE (%)": round(mdape, 2),
        "MAPE (%)": round(mape, 2),
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "R2": round(r2, 4),
    }


def disaggregate_metrics_by_segment(
    df_eval: pd.DataFrame,
    segment_col: str,
    y_true_col: str = "ClosePrice",
    y_pred_col: str = "PredClosePrice",
) -> pd.DataFrame:
    """Breaks down AVM performance across domain segments (e.g., price quintiles, age bands).

    Args:
        df_eval: DataFrame containing test set features, true close price, and predicted price.
        segment_col: Column name defining the slice/bin (e.g., 'PriceQuintile', 'AgeBand', 'County').
        y_true_col: Target column name.
        y_pred_col: Prediction column name.

    Returns:
        DataFrame summarizing sample counts and error metrics per segment, sorted by MdAPE.
    """
    results = []

    # Iterate over distinct categorical or binned segment groups
    for segment_name, group_df in df_eval.groupby(segment_col, observed=True):
        if len(group_df) == 0:
            continue

        # Evaluate localized metrics strictly on raw dollar targets
        metrics = calculate_avm_metrics(
            group_df[y_true_col].to_numpy(),
            group_df[y_pred_col].to_numpy(),
        )
        metrics[segment_col] = str(segment_name)
        metrics["SampleCount"] = len(group_df)
        results.append(metrics)

    res_df = pd.DataFrame(results)
    ordered_cols = [
        segment_col,
        "SampleCount",
        "MdAPE (%)",
        "MAPE (%)",
        "MAE ($)",
        "RMSE ($)",
        "R2",
    ]
    return res_df[ordered_cols].sort_values(by="MdAPE (%)")


def train_and_eval_boosting_model(
    model: Union[xgb.XGBRegressor, lgbm.LGBMRegressor],
    model_name: str,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Fits gradient boosting regressor on log-transformed target using validation early stopping.

    Args:
        model: Unfitted XGBoost or LightGBM regressor instance.
        model_name: Identifier string for tracking and logging.
        X_train, y_train: Training features and raw dollar targets (preceding X months).
        X_val, y_val: Chronological validation features and raw dollar targets ($Month_{N-1}$).
        X_test, y_test: Held-out test features and raw dollar targets ($Month_N$).

    Returns:
        Tuple of (multi-split summary metrics DataFrame, test predictions DataFrame for disaggregation).
    """
    # 1. Log-transform raw targets to enforce multiplicative percentage-loss optimization
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)

    # 2. Fit with chronological validation early stopping to prevent over-fitting tree depth/iterations
    if isinstance(model, xgb.XGBRegressor):
        model.fit(
            X_train,
            y_train_log,
            eval_set=[(X_val, y_val_log)],
            verbose=False,
        )
    elif isinstance(model, lgbm.LGBMRegressor):
        model.fit(
            X_train,
            y_train_log,
            eval_set=[(X_val, y_val_log)],
            callbacks=[lgbm.early_stopping(stopping_rounds=50, verbose=False)],
        )

    # 3. Invert log-space predictions back to raw dollar scale: y = exp(y_log) - 1
    y_train_pred = np.expm1(model.predict(X_train))
    y_val_pred = np.expm1(model.predict(X_val))
    y_test_pred = np.expm1(model.predict(X_test))

    # 4. Compute benchmark dollar-scale metrics across train, validation, and test partitions
    m_train = calculate_avm_metrics(y_train, y_train_pred)
    m_val = calculate_avm_metrics(y_val, y_val_pred)
    m_test = calculate_avm_metrics(y_test, y_test_pred)

    summary_df = pd.DataFrame(
        [m_train, m_val, m_test], index=["Train", "Validation", "Test"]
    )
    summary_df.insert(0, "Model", model_name)

    # 5. Build aligned prediction DataFrame for downstream error disaggregation & spatial profiling
    test_eval_df = X_test.copy()
    test_eval_df["ClosePrice"] = y_test
    test_eval_df["PredClosePrice"] = y_test_pred
    test_eval_df["Model"] = model_name

    return summary_df, test_eval_df


def run_week7_advanced_models(
    data_dir: Union[str, Path], output_dir: Union[str, Path]
) -> None:
    """Executes Week 7 Advanced Model pipeline: XGBoost & LightGBM training,

    hyperparameter comparison, and price quintile error disaggregation.
    """
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load chronologically partitioned features and raw dollar targets
    X_train = pd.read_parquet(data_dir / "X_train_processed.parquet")
    X_val = pd.read_parquet(data_dir / "X_val_processed.parquet")
    X_test = pd.read_parquet(data_dir / "X_test_processed.parquet")

    y_train = pd.read_parquet(data_dir / "y_train.parquet")["ClosePrice"].to_numpy()
    y_val = pd.read_parquet(data_dir / "y_val.parquet")["ClosePrice"].to_numpy()
    y_test = pd.read_parquet(data_dir / "y_test.parquet")["ClosePrice"].to_numpy()

    # 2. Instantiate advanced gradient boosting regressors with conservative regularization
    advanced_models = {
    "XGBoost_Tuned": xgb.XGBRegressor(
        # Boosting iterations budget; halted early via validation set to prevent tail overfitting
        n_estimators=1000,
        # Low shrinkage rate to steadily learn non-linear interactions across spatial and physical features
        learning_rate=0.03,
        # Depth 7 captures complex multi-way interactions (e.g., LivingArea x ZipCode x EffectiveAge) without over-segmenting micro-markets
        max_depth=7,
        # Row subsampling (80%) provides stochastic robustness against noisy monthly transaction spikes
        subsample=0.8,
        # Feature subsampling (70%) prevents dominant drivers (e.g., LivingArea) from masking subtle spatial/school signals
        colsample_bytree=0.7,
        # L1 regularization to zero out uninformative sparse features and noisy interaction flags
        reg_alpha=0.1,
        # L2 regularization to shrink weights across collinear spatial target encodings and geographic coordinates
        reg_lambda=1.0,
        # Fixed seed ensuring reproducible metrics across chronological backtesting cutoffs
        random_state=42,
        # Full CPU multi-threading for parallel histogram construction
        n_jobs=-1,
        # Halts fitting if validation metrics stall for 50 rounds (Requires `eval_set` in pipeline `.fit()`)
        early_stopping_rounds=50,
    ),
    "LightGBM_Tuned": lgbm.LGBMRegressor(
        # Boosting iterations budget paired with early stopping callback in pipeline execution
        n_estimators=1000,
        # Matches XGBoost shrinkage rate for fair cross-model backtesting comparison
        learning_rate=0.03,
        # Leaf-wise growth constraint (2^max_depth - 1 = 127); capped at 63 to enforce structural regularization
        num_leaves=63,
        # Hard limit on tree depth to prevent deep, asymmetric leaf splits on extreme luxury outliers
        max_depth=7,
        # Row subsampling fraction (80%) to reduce model variance
        subsample=0.8,
        # CRITICAL: Enables row bagging every 1 iteration; without `subsample_freq=1`, LightGBM silently IGNORES `subsample`
        subsample_freq=1,
        # Feature subsampling per split (feature_fraction) to encourage tree ensemble diversity
        colsample_bytree=0.7,
        # L1 regularization penalizing noisy high-cardinality categorical encodings (e.g., school district IDs)
        reg_alpha=0.1,
        # L2 regularization smoothing weight assignments across correlated physical property attributes
        reg_lambda=1.0,
        # Deterministic seed for reproducible out-of-fold target encoding and chronological validation
        random_state=42,
        # Multi-threading across available CPU cores for rapid leaf-wise spatial partitioning
        n_jobs=-1,
    ),
    }

    summary_list = []
    test_preds_dict = {}

    # Iterate over model suite, train on log-target, and extract predictions
    for name, model in advanced_models.items():
        summary_df, test_eval_df = train_and_eval_boosting_model(
            model, name, X_train, y_train, X_val, y_val, X_test, y_test
        )
        summary_list.append(summary_df)
        test_preds_dict[name] = test_eval_df

    master_summary = pd.concat(summary_list)
    print("=== Advanced Model Multi-Split Performance ===")
    print(master_summary.to_string())

    # 3. Disaggregate Test Set performance across domain bins (Example: Price Quintiles)
    # Using best model (XGBoost_Tuned) for disaggregation
    best_test_df = test_preds_dict["XGBoost_Tuned"]
    best_test_df["PriceQuintile"] = pd.qcut(
        best_test_df["ClosePrice"],
        q=5,
        labels=["Q1_Starter", "Q2_LowMid", "Q3_Median", "Q4_UpperMid", "Q5_Luxury"],
    )

    quintile_disaggregation = disaggregate_metrics_by_segment(
        best_test_df, segment_col="PriceQuintile"
    )

    print("\n=== Test Set Error Disaggregation (by Price Quintile) ===")
    print(quintile_disaggregation.to_string(index=False))

    # 4. Save evaluation metrics and disaggregated reports as CSV artifacts
    master_summary.to_csv(output_dir / "05_advanced_model_comparison.csv")
    quintile_disaggregation.to_csv(
        output_dir / "05_test_disaggregation_quintiles.csv", index=False
    )


if __name__ == "__main__":
    DATA_DIR = Path("model_artifacts")
    OUTPUT_DIR = Path("models/week7_advanced")
    run_week7_advanced_models(DATA_DIR, OUTPUT_DIR)

<>:105: SyntaxWarning: invalid escape sequence '\l'
<>:105: SyntaxWarning: invalid escape sequence '\l'
C:\Users\joeyw\AppData\Local\Temp\ipykernel_16536\37488799.py:105: SyntaxWarning: invalid escape sequence '\l'
  $y_{log} = \ln(1 + y)$ converts squared errors into relative multiplicative errors, directly
C:\Users\joeyw\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004292 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2022
[LightGBM] [Info] Number of data points in the train set: 57567, number of used features: 21
[LightGBM] [Info] Start training from score 13.745107
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain